In [20]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

sys.path.append("..")
from src import text_extraction, create_sentence_nace_code_similarities, analysis_functions
import test_base
from sentence_splitter import split_text_into_sentences

## Test retrieving the similarities for chunks in a pdf to the NACE Code

**Function:** pdf-> (chunk x code -> [-1,1])

**Parameters:** 

- pdf_path
- way of chunking the text (e.g. sentences, sliding window, or paragraphs)
- way of preprocessing (most is fixed for all reports)
    - similarity threshold of relevant chunks
    - length of irrelevant chunks

**Store analytics for each datapoint:**

- mean score for each class given a threshold

In [21]:
# Parameters: 

threshold_min_chunk_len = 100
cos_threshold = 0.4
sentence_length = 6

In [22]:
dataset_path = "../data/german_annual_reports"
dataset_path = "../data/stoxx_600_extended"
dataset_path = "../data/stoxx_600"

In [23]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [27]:
nace_classes = pd.read_csv(over_view_df_path, index_col=0)
nace_classes.head()

,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report
0,SalMar ASA,SALM-NO,SALM-NO,1984.965581,2458.824022,2271.61166349053,3.21,A,salmar-annual-report-2022.pdf
1,Bakkafrost P/F,BAKKA-NO,BAKKA-NO,929.503079,937.862170,973.053513491168,3.21,A,Bakkafrost PF2.pdf
2,Antofagasta plc,ANTO-GB,ANTO-GB,5577.681426,5849.975673,6113.94698310345,7.29,B,Antofagasta plc1.pdf
3,Anglo American plc,AAL-GB,AAL-GB,33423.271144,28355.894415,25288.1884924262,7.29,B,Anglo American plc1.pdf
4,TotalEnergies SE,TTE-FR,TTE-FR,250538.948328,202517.658053,180837.266896225,6.10,B,Totalenergies EP Gabon1.pdf


In [28]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class

{'salmar-annual-report-2022.pdf': 3.21,
 'Bakkafrost PF2.pdf': 3.21,
 'Antofagasta plc1.pdf': 7.29,
 'Anglo American plc1.pdf': 7.29,
 'Totalenergies EP Gabon1.pdf': 6.1,
 'subsea_2022-Annual-Report.pdf': 9.1,
 'shell-annual-report-2022.pdf': 6.1,
 'LANXESS AG1.pdf': 20.59,
 'Gerresheimer AG1.pdf': 22.22,
 'Verallia SAS3.pdf': 23.14,
 'BELIMO Holding AG1.pdf': 28.12,
 'Diageo PLC1.pdf': 11.01,
 'BAE Systems plc1.pdf': 30.3,
 'British American Tobacco p.l.c.1.pdf': 12.0,
 'Games Workshop Group PLC1.pdf': 32.4,
 'Greggs plc1.pdf': 10.71,
 'Halma plc1.pdf': 26.51,
 'Imperial Brands PLC3.pdf': 12.0,
 'IMI plc1.pdf': 28.99,
 'Howden Joinery Group PLC2.pdf': 27.51,
 'Associated British Foods plc1.pdf': 10.89,
 'Signify NV3.pdf': 27.4,
 'Smiths Group PLC1.pdf': 28.15,
 'Tate & Lyle PLC1.pdf': 10.89,
 'Smith & Nephew plc1.pdf': 32.5,
 'GSK PLC1.pdf': 21.2,
 'AstraZeneca PLC1.pdf': 21.2,
 'Burberry Group plc2.pdf': 14.19,
 'Airbus SE1.pdf': 30.3,
 "L'Oreal S.A.1.pdf": 20.42,
 'Dassault Aviation

In [29]:
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
report_to_nace_class

{'salmar-annual-report-2022.txt': 3.21,
 'Bakkafrost PF2.txt': 3.21,
 'Antofagasta plc1.txt': 7.29,
 'Anglo American plc1.txt': 7.29,
 'Totalenergies EP Gabon1.txt': 6.1,
 'subsea_2022-Annual-Report.txt': 9.1,
 'shell-annual-report-2022.txt': 6.1,
 'LANXESS AG1.txt': 20.59,
 'Gerresheimer AG1.txt': 22.22,
 'Verallia SAS3.txt': 23.14,
 'BELIMO Holding AG1.txt': 28.12,
 'Diageo PLC1.txt': 11.01,
 'BAE Systems plc1.txt': 30.3,
 'British American Tobacco p.l.c.1.txt': 12.0,
 'Games Workshop Group PLC1.txt': 32.4,
 'Greggs plc1.txt': 10.71,
 'Halma plc1.txt': 26.51,
 'Imperial Brands PLC3.txt': 12.0,
 'IMI plc1.txt': 28.99,
 'Howden Joinery Group PLC2.txt': 27.51,
 'Associated British Foods plc1.txt': 10.89,
 'Signify NV3.txt': 27.4,
 'Smiths Group PLC1.txt': 28.15,
 'Tate & Lyle PLC1.txt': 10.89,
 'Smith & Nephew plc1.txt': 32.5,
 'GSK PLC1.txt': 21.2,
 'AstraZeneca PLC1.txt': 21.2,
 'Burberry Group plc2.txt': 14.19,
 'Airbus SE1.txt': 30.3,
 "L'Oreal S.A.1.txt": 20.42,
 'Dassault Aviation

In [30]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
reports_path

['../data/stoxx_600/TXTs/Hannover Rueck SE1.txt',
 '../data/stoxx_600/TXTs/Interpump Group S.p.A.1.txt',
 '../data/stoxx_600/TXTs/Intertek Group PLC1.txt',
 '../data/stoxx_600/TXTs/Anheuser-Busch InBev SANV3.txt',
 '../data/stoxx_600/TXTs/Scout24 SE3.txt',
 '../data/stoxx_600/TXTs/Bridgepoint Group Plc1.txt',
 '../data/stoxx_600/TXTs/Financiere de Tubize SA2.txt',
 '../data/stoxx_600/TXTs/Severn Trent Plc1.txt',
 '../data/stoxx_600/TXTs/Bakkafrost PF2.txt',
 '../data/stoxx_600/TXTs/Ferrari NV2.txt',
 '../data/stoxx_600/TXTs/Sika AG3.txt',
 '../data/stoxx_600/TXTs/Swiss Prime Site AG2.txt',
 '../data/stoxx_600/TXTs/Poste Italiane SpA2.txt',
 '../data/stoxx_600/TXTs/Haleon PLC1.txt',
 '../data/stoxx_600/TXTs/Land Securities Group PLC2.txt',
 '../data/stoxx_600/TXTs/Rentokil Initial plc2.txt',
 '../data/stoxx_600/TXTs/Nexans SA3.txt',
 '../data/stoxx_600/TXTs/SEB SA2.txt',
 '../data/stoxx_600/TXTs/Deutsche Bank Aktiengesellschaft1.txt',
 '../data/stoxx_600/TXTs/Technip Energies NV1.txt',


In [31]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [34]:
for i in range(1): 
    nace_level = i

    result_path = f"../results/dataset__{dataset_name}_sentence_len_{sentence_length}__min_chunk_len_{threshold_min_chunk_len}__cos_thresh_{cos_threshold}__nace_level_{nace_level}"

    res = test_base.test_similarities(reports_path, preprocess_report, threshold_min_chunk_len, cos_threshold, report_to_nace_class, result_path, level=i)

  0%|          | 0/13 [00:00<?, ?it/s]

Report:  ../data/stoxx_600/TXTs/Melrose Industries PLC1.txt
Number of Chunks:  2138


  8%|▊         | 1/13 [00:15<03:05, 15.46s/it]

Report:  ../data/stoxx_600/TXTs/Balfour Beatty plc1.txt
Number of Chunks:  2045


  8%|▊         | 1/13 [00:22<04:29, 22.44s/it]


KeyboardInterrupt: 

In [ ]:
# test for different sentence lengths

# nace_level = 1
# for i in [3,4,5,6,7]: 

#     sentence_length = i

#     def preprocess_report(pdf_path: str) -> List[str]:

#         with open(pdf_path, "r") as f: 
#             text = f.read()
        
#         lines = text.split("\n")

#         # drop if condidtion is True
#         conditions = [
#             # filter images
#             lambda line: line == '<!-- image -->',
            
#             #filter tables 
#             lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

#             # filter headers
#             lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

#             # filter sentences
#             lambda line: "." not in line,
            
#             # more than 50% is numbers
#             lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

#             # minimum 3 words 
#             lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

#             # Minimum 2 Sentences
#             #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

#         ]
#         accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]

#         chunks = []

#         for line in accepted_lines: 
#             sentences = split_text_into_sentences(line, language='en')
#             sentences = [sentence.strip() for sentence in sentences]
#             sentences = [sentence for sentence in sentences if sentence != ""]
#             new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

#             chunks += new_chunks

#         # if there is only one sentence in the last chunk, balance the two last chunks
#         if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
#             last_two_chunks = chunks[-2] + " " + chunks[-1]
#             chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
#             chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

#         chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
#         chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
#         chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
#         chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
#         chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
#         chunks = [chunk.lower() for chunk in chunks]
#         chunks = [chunk.strip() for chunk in chunks]

#         return chunks
    
#     result_path = f"../results/paragraph_and_sentence_len_{sentence_length}_min_chunk_len_{threshold_min_chunk_len}_cos_thresh_{cos_threshold}_nace_level_{nace_level}_stoxx"

#     res = test_base.test_similarities(reports_path, preprocess_report, threshold_min_chunk_len, cos_threshold, report_to_nace_class, result_path, level=i)